# From-Scratch K-Nearest Neighbors Classifier

This cleaned notebook demonstrates the KNN implementation using reusable code from the `src/` folder. It keeps the analysis reproducible by using fixed `k` values instead of interactive input prompts.


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT / "src"))

import pandas as pd

from data import read_xy, validate_xy, train_test_split, class_distribution
from knn import KNNClassifier
from metrics import accuracy_score, binary_metrics, confusion_matrix
from visualization import plot_confusion_matrix, plot_accuracy_by_k

DATA_DIR = PROJECT_ROOT / "data" / "raw"


## Binary classification experiment


In [ ]:
binary_xy = read_xy(DATA_DIR / "Prog1data.xlsx", header=None)
binary_xy_clean = validate_xy(binary_xy, {"+", "-"})

print("Original shape:", binary_xy.shape)
print("Cleaned shape:", binary_xy_clean.shape)
print("Rows removed:", binary_xy.shape[0] - binary_xy_clean.shape[0])

class_distribution(binary_xy_clean[:, -1], ["+", "-"])


In [ ]:
x_train, x_test, y_train, y_test = train_test_split(binary_xy_clean, train_ratio=0.8)

rows = []
for k in [5, 15, 25, 35]:
    model = KNNClassifier(k=k).fit(x_train, y_train)
    predictions = model.predict(x_test)
    metrics = binary_metrics(y_test, predictions)
    rows.append({"k": k, **{name: metrics[name] for name in ["accuracy", "precision", "recall", "f1"]}})

binary_results = pd.DataFrame(rows)
binary_results


In [ ]:
model = KNNClassifier(k=5).fit(x_train, y_train)
predictions = model.predict(x_test)
cm = confusion_matrix(y_test, predictions, labels=["+", "-"])
plot_confusion_matrix(cm, labels=["+", "-"], title="Binary KNN Confusion Matrix, k=5");


## Multiclass Iris experiment


In [ ]:
iris_xy = read_xy(DATA_DIR / "Iris.xlsx", header=0)[:, 1:]
iris_labels = ["Iris-setosa", "Iris-versicolor", "Iris-virginica"]
iris_xy_clean = validate_xy(iris_xy, iris_labels)

print("Original shape:", iris_xy.shape)
print("Cleaned shape:", iris_xy_clean.shape)
class_distribution(iris_xy_clean[:, -1], iris_labels)


In [ ]:
x_train, x_test, y_train, y_test = train_test_split(iris_xy_clean, train_ratio=0.8)

k_values = list(range(5, 100, 10))
train_accuracy = []
test_accuracy = []

for k in k_values:
    model = KNNClassifier(k=k).fit(x_train, y_train)
    train_accuracy.append(accuracy_score(y_train, model.predict(x_train)))
    test_accuracy.append(accuracy_score(y_test, model.predict(x_test)))

iris_results = pd.DataFrame({"k": k_values, "train_accuracy": train_accuracy, "test_accuracy": test_accuracy})
iris_results


In [ ]:
plot_accuracy_by_k(k_values, train_accuracy, test_accuracy);


## Takeaways

The implementation shows how KNN performs well on small, low-dimensional datasets but becomes computationally expensive because prediction requires comparing each new point to every training point. The full written discussion is available in `reports/project_report.md`.
